# Uncertainty intervals for the five-league evaluation

This notebook measures how uncertain the model comparisons are. It resamples complete match weeks within every league and season, keeping the two models paired on identical matches.

The primary test is the pooled recalibrated market versus the pooled market-plus-player model. Equal-league results are primary; match-weighted results are secondary. Positive improvement means the enhanced model is better. The notebook uses development seasons only and does not access 2025/26.

In [4]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'scotland_research').is_dir()
)
RESEARCH_DIR = PROJECT_ROOT / 'scotland_research'
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

from constants import DEFAULT_EVALUATION_DIR
from evaluation.report import write_csv_atomic
from selected_features import load_selected_features, validate_selected_features
from uncertainty import (
    prepare_paired_comparison,
    resample_league_season_weeks,
    summarize_interval,
)

EVALUATION_DIR = DEFAULT_EVALUATION_DIR
OUTPUT_DIR = RESEARCH_DIR / 'visuals' / 'uncertainty_measurements'
TABLES_DIR = OUTPUT_DIR / 'tables'
FIGURES_DIR = OUTPUT_DIR / 'figures'
REPETITIONS = 10_000
SEED = 42

print(f'Evaluation input: {EVALUATION_DIR}')
print(f'Outputs: {OUTPUT_DIR}')

Evaluation input: C:\Users\skous\Super-League-odds-research\artifacts\five_league_development_evaluation
Outputs: C:\Users\skous\Super-League-odds-research\scotland_research\visuals\uncertainty_measurements


## Comparisons

The first comparison directly answers the research question. The remaining comparisons separate market recalibration, league-specific player effects and Dixon–Coles player effects.

In [5]:
COMPARISONS = [
    {
        'comparison': 'Pooled players versus recalibrated market',
        'priority': 'primary',
        'training_scope': 'pooled',
        'baseline_model': 'recalibrated_market',
        'enhanced_model': 'market_plus_player_form',
    },
    {
        'comparison': 'Pooled recalibration versus raw market',
        'priority': 'secondary',
        'training_scope': 'pooled',
        'baseline_model': 'closing_market',
        'enhanced_model': 'recalibrated_market',
    },
    {
        'comparison': 'Separate players versus recalibrated market',
        'priority': 'secondary',
        'training_scope': 'league_specific',
        'baseline_model': 'recalibrated_market',
        'enhanced_model': 'market_plus_player_form',
    },
    {
        'comparison': 'Dixon-Coles with players versus plain Dixon-Coles',
        'priority': 'secondary',
        'training_scope': 'league_specific',
        'baseline_model': 'dixon_coles',
        'enhanced_model': 'dixon_coles_player_form',
    },
]
pd.DataFrame(COMPARISONS)

,comparison,priority,training_scope,baseline_model,enhanced_model
0,Pooled players versus recalibrated market,primary,pooled,recalibrated_market,market_plus_player_form
1,Pooled recalibration versus raw market,secondary,pooled,closing_market,recalibrated_market
2,Separate players versus recalibrated market,secondary,league_specific,recalibrated_market,market_plus_player_form
3,Dixon-Coles with players versus plain Dixon-Coles,secondary,league_specific,dixon_coles,dixon_coles_player_form


## Load and verify the current evaluation

The selected-feature copy saved beside the predictions must exactly match the active fixed specification. If the feature list changed after the evaluation, this cell stops and asks for a rerun instead of analysing stale predictions.

In [6]:
predictions_path = EVALUATION_DIR / 'predictions.csv'
evaluated_features_path = EVALUATION_DIR / 'selected_features.csv'
missing_files = [
    str(path)
    for path in (predictions_path, evaluated_features_path)
    if not path.exists()
]
if missing_files:
    raise FileNotFoundError(
        'Run scotland_research/evaluate_models.py first. Missing: '
        + ', '.join(missing_files)
    )

current_features = load_selected_features()
try:
    _, evaluated_hash = validate_selected_features(
        pd.read_csv(evaluated_features_path, dtype='string')
    )
except ValueError as error:
    raise RuntimeError(
        'The saved evaluation does not use the current selected features. '
        'Rerun scotland_research/evaluate_models.py before measuring uncertainty.'
    ) from error
if evaluated_hash != current_features.semantic_sha256:
    raise RuntimeError(
        'The saved evaluation feature checksum is stale. '
        'Rerun scotland_research/evaluate_models.py.'
    )

predictions = pd.read_csv(predictions_path)
available = predictions.groupby(['training_scope', 'model']).size().rename('matches')
print(f'Loaded {len(predictions):,} prediction rows.')
print(f'Feature checksum: {evaluated_hash}')
available

Loaded 31,160 prediction rows.
Feature checksum: 2f3f3a757a26bcd3df93eb798ae8dd8135d82bb40ab4c695a39f0846c4655df4


training_scope   model                  
league_specific  closing_market             3895
                 dixon_coles                3895
                 dixon_coles_player_form    3895
                 market_plus_player_form    3895
                 recalibrated_market        3895
pooled           closing_market             3895
                 market_plus_player_form    3895
                 recalibrated_market        3895
Name: matches, dtype: int64

## Paired match-week bootstrap

For each comparison, the notebook calculates each match's log loss, Brier score and normalized RPS under both models. It then resamples match-week blocks with replacement inside every league-season. Both models always receive the same sampled matches.

In [ ]:
summary_frames = []
sample_frames = []
match_audit_rows = []

for comparison_number, comparison in enumerate(COMPARISONS):
    paired = prepare_paired_comparison(
        predictions,
        training_scope=comparison['training_scope'],
        baseline_model=comparison['baseline_model'],
        enhanced_model=comparison['enhanced_model'],
    )
    comparison_seed = SEED + comparison_number
    samples = resample_league_season_weeks(
        paired,
        repetitions=REPETITIONS,
        seed=comparison_seed,
    )
    summary_frames.append(
        summarize_interval(
            paired,
            samples,
            comparison,
            repetitions=REPETITIONS,
            seed=comparison_seed,
        )
    )
    samples.insert(0, 'comparison', comparison['comparison'])
    sample_frames.append(samples)
    match_audit_rows.append(
        {
            **comparison,
            'matches': len(paired),
            'leagues': paired['league'].nunique(),
            'seasons': paired['season'].nunique(),
            'league_season_weeks': paired[
                ['league', 'season', 'match_week']
            ].drop_duplicates().shape[0],
        }
    )
    print(f"Finished: {comparison['comparison']}")

summary = pd.concat(summary_frames, ignore_index=True)
bootstrap_samples = pd.concat(sample_frames, ignore_index=True)
match_audit = pd.DataFrame(match_audit_rows)

Finished: Pooled players versus recalibrated market


## Primary result

An interval entirely above zero supports the enhanced model. An interval containing zero means the development data do not clearly distinguish the models.

In [ ]:
primary = summary[
    summary['priority'].eq('primary')
    & summary['weighting'].eq('equal_league')
][
    [
        'metric',
        'matches',
        'observed_absolute_improvement',
        'lower_95_absolute',
        'upper_95_absolute',
        'observed_relative_improvement_pct',
        'lower_95_relative_pct',
        'upper_95_relative_pct',
        'samples_favouring_enhanced_pct',
        'interval_excludes_zero',
    ]
].copy()
primary

## All comparisons

In [ ]:
summary.sort_values(
    ['priority', 'comparison', 'weighting', 'metric'],
    kind='stable',
)[
    [
        'comparison',
        'weighting',
        'metric',
        'observed_relative_improvement_pct',
        'lower_95_relative_pct',
        'upper_95_relative_pct',
        'samples_favouring_enhanced_pct',
        'interval_excludes_zero',
    ]
]

## Interval figure

The dots are observed equal-league improvements. Horizontal lines show the middle 95% of the paired bootstrap results.

In [ ]:
plot_data = summary[summary['weighting'].eq('equal_league')].copy()
comparison_order = [item['comparison'] for item in COMPARISONS]
metric_labels = {
    'log_loss': 'Log loss',
    'brier_score': 'Brier score',
    'rps': 'RPS',
}
fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True, constrained_layout=True)
for axis, (metric, metric_label) in zip(axes, metric_labels.items()):
    current = (
        plot_data[plot_data['metric'].eq(metric)]
        .set_index('comparison')
        .loc[comparison_order]
        .reset_index()
    )
    y = np.arange(len(current))
    observed = current['observed_absolute_improvement'].to_numpy()
    lower = current['lower_95_absolute'].to_numpy()
    upper = current['upper_95_absolute'].to_numpy()
    colors = ['#2166ac' if priority == 'primary' else '#666666' for priority in current['priority']]
    axis.errorbar(
        observed,
        y,
        xerr=np.vstack([observed - lower, upper - observed]),
        fmt='none',
        ecolor=colors,
        elinewidth=2,
        capsize=4,
    )
    axis.scatter(observed, y, c=colors, s=45, zorder=3)
    axis.axvline(0, color='black', linestyle='--', linewidth=1)
    axis.set_title(metric_label)
    axis.set_xlabel('Improvement over baseline')
    axis.spines[['top', 'right']].set_visible(False)
    axis.set_yticks(y, comparison_order)
    axis.invert_yaxis()
fig.suptitle('Paired match-week uncertainty intervals — equal-league results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / 'uncertainty_intervals.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {figure_path}')

## Save auditable outputs

In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
write_csv_atomic(summary, TABLES_DIR / 'uncertainty_intervals.csv')
write_csv_atomic(match_audit, TABLES_DIR / 'comparison_match_audit.csv')

settings = {
    'development_only': True,
    'prediction_source': str(predictions_path),
    'selected_features_sha256': evaluated_hash,
    'repetitions': REPETITIONS,
    'base_seed': SEED,
    'resampling_unit': 'match week within league and season',
    'primary_weighting': 'equal league',
    'secondary_weighting': 'match weighted',
    'primary_metric': 'log loss',
    'secondary_metrics': ['Brier score', 'normalized RPS'],
    'comparisons': COMPARISONS,
}
settings_path = TABLES_DIR / 'run_settings.json'
settings_path.write_text(json.dumps(settings, indent=2), encoding='utf-8')
print(f'Saved interval tables and settings to {TABLES_DIR}')

## Interpretation rules

- Focus first on the equal-league log-loss interval for **Pooled players versus recalibrated market**.
- If its 95% interval includes zero, describe the player improvement as uncertain rather than meaningful.
- Brier score and RPS should point in the same general direction; they are supporting measures, not reasons to override log loss.
- Match-weighted results show whether fixture-heavy leagues change the conclusion.
- Secondary comparisons explain the model behaviour but do not replace the primary research test.
- These intervals describe development-sample uncertainty. The untouched 2025/26 evaluation remains the final independent test.